In [1]:
pip install protobuf==3.20.3


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mlflow
import mlflow.keras   # or mlflow.pytorch if you use PyTorch

# Replace with your actual DagsHub username and repo name
import dagshub
dagshub.init(
    repo_owner="ahmed.adel1711",
    repo_name="SLR-Main",
    mlflow=True
)
# This one line connects MLflow to the cloud. Nothing stored locally.

Accessing as hadeelgamal.sis.2020

Initialized MLflow to track repo "ahmed.adel1711/SLR-Main"

Repository ahmed.adel1711/SLR-Main initialized!

In [ ]:
with mlflow.start_run(run_name="MediaPipe_training_V2"):

    # --- LOG ARCHITECTURE & MODEL CONFIG ---
    mlflow.log_param("architecture", "MLP")
    mlflow.log_param("landmark_type", "mediapipe_hands")
    mlflow.log_param("input_features", 63)           # 21 landmarks × (x, y, z)
    mlflow.log_param("output_classes", 32)            # 32 Arabic letters
    mlflow.log_param("layers", "512→256→64→32")
    mlflow.log_param("activation", "relu")
    mlflow.log_param("output_activation", "softmax")
    mlflow.log_param("kernel_initializer", "he_normal")
    mlflow.log_param("l2_regularization", 1e-4)       # applied on dense_512 & dense_256
    mlflow.log_param("dropout_rate", 0.2)             # all 3 dropout layers
    mlflow.log_param("batch_normalization", True)
    mlflow.log_param("total_params", 185696)
    mlflow.log_param("trainable_params", 184160)

    # --- LOG TRAINING CONFIG ---
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.0005)         # actual code value (not 0.001 in print)
    mlflow.log_param("loss", "categorical_crossentropy")
    mlflow.log_param("batch_size", 256)
    mlflow.log_param("epochs_max", 100)
    mlflow.log_param("early_stopping_patience", 15)
    mlflow.log_param("reduce_lr_factor", 0.5)
    mlflow.log_param("reduce_lr_patience", 7)
    mlflow.log_param("min_lr", 1e-7)
    mlflow.log_param("mixed_precision", "mixed_float16")

    # --- LOG DATA CONFIG ---
    mlflow.log_param("dataset", "ASLAD-3000")
    mlflow.log_param("max_images_per_class", 4000)
    mlflow.log_param("mediapipe_min_confidence", 0.5)
    mlflow.log_param("mediapipe_model_complexity", 0)
    mlflow.log_param("train_split", 0.60)
    mlflow.log_param("val_split", 0.20)
    mlflow.log_param("test_split", 0.20)
    mlflow.log_param("random_state", 42)

    # --- TRAIN YOUR MODEL (your existing code) ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=100,
        callbacks=callbacks,
        verbose=1
    )

    # --- LOG METRICS PER EPOCH ---
    for epoch, (loss, acc, val_loss, val_acc) in enumerate(zip(
            history.history['loss'],
            history.history['accuracy'],
            history.history['val_loss'],
            history.history['val_accuracy'])):
        mlflow.log_metric("train_loss",     loss,     step=epoch)
        mlflow.log_metric("train_accuracy", acc,      step=epoch)
        mlflow.log_metric("val_loss",       val_loss, step=epoch)
        mlflow.log_metric("val_accuracy",   val_acc,  step=epoch)

    # --- LOG FINAL TEST EVAL METRICS ---
    mlflow.log_metric("test_loss",               0.0578)
    mlflow.log_metric("test_accuracy",           0.9945)   # 99.45%
    mlflow.log_metric("test_samples",            14734)
    mlflow.log_metric("evaluation_time_seconds", 1.3801)

    # --- SAVE ARTIFACTS ---
    mlflow.log_artifact("arsl_mediapipe_mlp_model_best.h5")
    mlflow.log_artifact("arsl_mediapipe_mlp_model_v2.2.h5")
    mlflow.log_artifact("confusion_matrix.png")           # if you generate one

    # --- LOG THE MODEL ITSELF ---
    mlflow.keras.log_model(model, "model")